<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>

# Assignment: SQL Notebook for Peer Assignment

**Author:** Ahmad Waziri

Using this notebook we:
1. Load the `Spacex.csv` dataset into a SQLite database
2. Execute SQL queries to answer the assignment questions about SpaceX launch history

*Note: this notebook uses the `sqlite3` module directly (with pandas' `read_sql_query` to render
results) instead of the `ipython-sql`/`%sql` magic extension, since the magic extension requires
an interactive IPython kernel with a database driver that isn't available in this execution
environment. The SQL itself, and the results, are identical either way.*

In [1]:
import pandas as pd
import sqlite3

con = sqlite3.connect("my_data1.db")
cur = con.cursor()

In [1]:
df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv")
df.to_sql("SPACEXTBL", con, if_exists='replace', index=False, method="multi")

101

**Note:** the code below removes blank rows from the table.

In [1]:
cur.execute("DROP TABLE IF EXISTS SPACEXTABLE;")
cur.execute("CREATE TABLE SPACEXTABLE AS SELECT * FROM SPACEXTBL WHERE Date IS NOT NULL;")
con.commit()

## Tasks

Now write and execute SQL queries to solve the assignment tasks.

*Note: If the column names are in mixed case, enclose them in double quotes.*

### Task 1

##### Display the names of the unique launch sites in the space mission

In [1]:
pd.read_sql_query("SELECT DISTINCT Launch_Site FROM SPACEXTABLE;", con)

    Launch_Site
0   CCAFS LC-40
1   VAFB SLC-4E
2    KSC LC-39A
3  CCAFS SLC-40

### Task 2

##### Display 5 records where launch sites begin with the string 'CCA'

In [1]:
pd.read_sql_query("SELECT * FROM SPACEXTABLE WHERE Launch_Site LIKE 'CCA%' LIMIT 5;", con)

         Date Time (UTC) Booster_Version  Launch_Site                                                        Payload  PAYLOAD_MASS__KG_      Orbit         Customer Mission_Outcome      Landing_Outcome
0  2010-06-04   18:45:00  F9 v1.0  B0003  CCAFS LC-40                           Dragon Spacecraft Qualification Unit                  0        LEO           SpaceX         Success  Failure (parachute)
1  2010-12-08   15:43:00  F9 v1.0  B0004  CCAFS LC-40  Dragon demo flight C1, two CubeSats, barrel of Brouere cheese                  0  LEO (ISS)  NASA (COTS) NRO         Success  Failure (parachute)
2  2012-05-22    7:44:00  F9 v1.0  B0005  CCAFS LC-40                                          Dragon demo flight C2                525  LEO (ISS)      NASA (COTS)         Success           No attempt
3  2012-10-08    0:35:00  F9 v1.0  B0006  CCAFS LC-40                                                   SpaceX CRS-1                500  LEO (ISS)       NASA (CRS)         Success           No att

### Task 3

##### Display the total payload mass carried by boosters launched by NASA (CRS)

In [1]:
pd.read_sql_query(
    "SELECT SUM(PAYLOAD_MASS__KG_) AS total_payload_mass_kg FROM SPACEXTABLE WHERE Customer = 'NASA (CRS)';",
    con
)

   total_payload_mass_kg
0                  45596

### Task 4

##### Display average payload mass carried by booster version F9 v1.1

In [1]:
pd.read_sql_query(
    "SELECT AVG(PAYLOAD_MASS__KG_) AS avg_payload_mass_kg FROM SPACEXTABLE WHERE Booster_Version = 'F9 v1.1';",
    con
)

   avg_payload_mass_kg
0               2928.4

### Task 5

##### List the date when the first successful landing outcome on a ground pad was achieved

_Hint: use the MIN function_

In [1]:
pd.read_sql_query(
    "SELECT MIN(Date) AS first_ground_pad_success FROM SPACEXTABLE WHERE Landing_Outcome = 'Success (ground pad)';",
    con
)

  first_ground_pad_success
0               2015-12-22

### Task 6

##### List the names of the boosters which have success in drone ship and have payload mass greater than 4000 but less than 6000

In [1]:
pd.read_sql_query(
    """SELECT Booster_Version FROM SPACEXTABLE
       WHERE Landing_Outcome = 'Success (drone ship)'
       AND PAYLOAD_MASS__KG_ BETWEEN 4000 AND 6000;""",
    con
)

  Booster_Version
0     F9 FT B1022
1     F9 FT B1026
2  F9 FT  B1021.2
3  F9 FT  B1031.2

### Task 7

##### List the total number of successful and failure mission outcomes

In [1]:
pd.read_sql_query(
    """SELECT
           SUM(CASE WHEN Mission_Outcome LIKE 'Success%' THEN 1 ELSE 0 END) AS total_success,
           SUM(CASE WHEN Mission_Outcome LIKE 'Failure%' THEN 1 ELSE 0 END) AS total_failure
       FROM SPACEXTABLE;""",
    con
)

   total_success  total_failure
0            100              1

### Task 8

##### List all the booster_versions that carried the maximum payload mass, using a subquery

In [1]:
pd.read_sql_query(
    """SELECT DISTINCT Booster_Version FROM SPACEXTABLE
       WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTABLE);""",
    con
)

   Booster_Version
0    F9 B5 B1048.4
1    F9 B5 B1049.4
2    F9 B5 B1051.3
3    F9 B5 B1056.4
4    F9 B5 B1048.5
5    F9 B5 B1051.4
6    F9 B5 B1049.5
7   F9 B5 B1060.2 
8   F9 B5 B1058.3 
9    F9 B5 B1051.6
10   F9 B5 B1060.3
11  F9 B5 B1049.7 

### Task 9

##### List the records which display the month, failure landing_outcomes in drone ship, booster versions, launch_site for the months in year 2015

**Note:** SQLite does not support month names, so we use `substr(Date, 6, 2)` for month and
`substr(Date, 0, 5) = '2015'` for year.

In [1]:
pd.read_sql_query(
    """SELECT substr(Date,6,2) AS month, Landing_Outcome, Booster_Version, Launch_Site
       FROM SPACEXTABLE
       WHERE substr(Date,0,5) = '2015' AND Landing_Outcome = 'Failure (drone ship)';""",
    con
)

  month       Landing_Outcome Booster_Version  Launch_Site
0    01  Failure (drone ship)   F9 v1.1 B1012  CCAFS LC-40
1    04  Failure (drone ship)   F9 v1.1 B1015  CCAFS LC-40

### Task 10

##### Rank the count of landing outcomes between 2010-06-04 and 2017-03-20, in descending order

In [1]:
pd.read_sql_query(
    """SELECT Landing_Outcome, COUNT(*) AS outcome_count
       FROM SPACEXTABLE
       WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
       GROUP BY Landing_Outcome
       ORDER BY outcome_count DESC;""",
    con
)

          Landing_Outcome  outcome_count
0              No attempt             10
1    Success (drone ship)              5
2    Failure (drone ship)              5
3    Success (ground pad)              3
4      Controlled (ocean)              3
5    Uncontrolled (ocean)              2
6     Failure (parachute)              2
7  Precluded (drone ship)              1

## Conclusion

NASA's CRS cargo missions and the plain F9 v1.1 booster variant each show distinctive payload
profiles, the first successful ground-pad landing was the historic OG2 Mission 2 flight, and the
heaviest payloads in the dataset belong to SpaceX's own reused Starlink missions on Block 5
boosters. Landing recovery outcomes improved markedly over time, with 'No attempt' and early
drone-ship failures concentrated in 2015-2016 before SpaceX's landing success rate climbed.